## Self attention

In [2]:
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [3]:
attention_scores = inputs @ inputs.T
print(attention_scores)
print(attention_scores.shape)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
torch.Size([6, 6])


In [4]:
attention_weights = torch.softmax(attention_scores,dim=1)
print(attention_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [5]:
context_vector = attention_weights @ inputs
print(context_vector)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


# W_v, W_q, W_k matrices 
Using W_v, W_q, W_k as learnable parameters for context vector creation

In [6]:
import torch
import torch.nn as nn

class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias= qkv_bias)
    def forward(self, x):
        query = self.W_query(x)
        key = self.W_key(x)
        value = self.W_value(x)
        attention_scores = query @ key.T
        attention_weights = torch.softmax((attention_scores/(key.shape[-1]**0.5)), dim=-1)
        context_vectors = attention_weights @ value
        return context_vectors

In [7]:
self_attention = SelfAttention(d_in = 3, d_out = 2)
context_vectors = self_attention(inputs)
print(context_vectors)

tensor([[0.3646, 0.0913],
        [0.3650, 0.0910],
        [0.3650, 0.0910],
        [0.3630, 0.0918],
        [0.3638, 0.0915],
        [0.3632, 0.0917]], grad_fn=<MmBackward0>)


# Causal Attention


In [8]:
context_length = 6
keys_dim = 6
mask = torch.triu(torch.ones(context_length, context_length), diagonal =1)
masked = attention_scores.masked_fill(mask.bool(), -torch.inf)
attn_weights = torch.softmax(masked / keys_dim**0.5, dim=1)
print(mask)
print(masked)
print(attn_weights)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.9995,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.9544, 1.4950,   -inf,   -inf,   -inf,   -inf],
        [0.9422, 1.4754, 1.4570,   -inf,   -inf,   -inf],
        [0.4753, 0.8434, 0.8296, 0.4937,   -inf,   -inf],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654,   -inf],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4450, 0.5550, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2876, 0.3575, 0.3549, 0.0000, 0.0000, 0.0000],
        [0.2312, 0.2687, 0.2672, 0.2329, 0.0000, 0.0000],
        [0.1900, 0.2104, 0.2111, 0.1817, 0.2068, 0.0000],
        [0.1559, 0.1878, 0.1858, 0.1575, 0.1358, 0.1772]])


In [9]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)   

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length,
                dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)            #1
        self.register_buffer(
           'mask',
           torch.triu(torch.ones(context_length, context_length),
           diagonal=1)
        )             #2

    def forward(self, x):
        b, num_tokens, d_in = x.shape                   #3
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)   
        attn_scores.masked_fill_(                    #4
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out  = 3, 2
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 6, 3])
context_vecs.shape: torch.Size([2, 6, 2])


## Dropout Scaling 
Each remaining output is scaled by 1/(1-p)

In [10]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.1)    #1
example = torch.ones(6, 6)      #2
print(dropout(example))


tensor([[1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 0.0000, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 0.0000, 1.1111, 1.1111, 1.1111],
        [1.1111, 1.1111, 1.1111, 1.1111, 1.1111, 1.1111]])


# Multihead Attention

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout, qkv_bias= False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"
        self.head_dim = d_out // num_heads
        self.num_heads = num_heads
        self.d_out = d_out
        self.W_key= nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_query= nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_value= nn.Linear(d_in, d_out, bias= qkv_bias)
        self.out_project = nn.Linear(d_out, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal =1))

    def forward(self, x):
        b, num_tokens, emd_dim = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        attention_scores = queries @ keys.transpose(2,3)
        attention_scores.masked_fill_( self.mask.bool()[ :num_tokens, :num_tokens], -torch.inf)
        attention_weights = torch.softmax( attention_scores / keys.shape[-1]**0.5, dim= -1)
        attention_weights = dropout(attention_weights)
        context_vectors = (attention_weights @ values)
        context_vectors = context_vectors.contiguous().view(
            b, num_tokens, self.d_out
        )
        context_vectors = self.out_project(context_vectors)    #11
        return context_vectors

        
        

In [12]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length= context_length, dropout= 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.0630, -0.3363],
         [-0.0268, -0.4303],
         [-0.0111, -0.3714],
         [-0.0133,  0.1392],
         [-0.0229, -0.0081],
         [-0.0100, -0.0689]],

        [[-0.0630, -0.3363],
         [ 0.0184, -0.4040],
         [-0.0581, -0.3031],
         [-0.0382,  0.1248],
         [-0.0117, -0.0453],
         [-0.0100, -0.0689]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


# GPT model

In [13]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,          # Embedding dimension
    "n_heads": 12,           # Number of attention heads
    "n_layers": 12,          # Number of layers
    "drop_rate": 0.1,        # Dropout rate
    "qkv_bias": False        # Query-Key-Value bias
}

In [14]:
import torch.nn as nn
import torch

cfg = GPT_CONFIG_124M

class DummyGPTModel():
    def __init__(self, cfg):
        super.__init__()
        self.tok_emb = nn.Linear(cfg["vocab_size"], cfg["emb_dim"], bias= False)
        self.pos_emb = nn.Linear(cfg["context_length"], cfg["emb_dim"], bias = False)
        self.mha = MultiHeadAttention(cfg["emd_dim"], cfg["emb_dim"], cfg["contex_length"], cfg["drop_rate"])
        self.dummy_transformer_block = nn.Sequential( *[transformer for _ in range(cfg["n_layers"])])
        self.out_layer = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias= False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx
        x = self.tok_emb(in_idx) + self.pos_emb(in_idx)
        x = self.dummy_transformer_block(x)
        logits = self.out_layer(x)
        probs = torch.softmax(logits, dim=-1, keepdim = True)
        next_word = vocab[torch.argmax(logits)]
        return next_word
        
        
        

In [15]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift



![description](images/gptmodel.png)

In [16]:

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
 #1
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut      #2

        shortcut = x         #3
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut      #4
        return x

In [17]:
torch.manual_seed(123)
x = torch.rand(2, 4, 768)                   #1
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


In [18]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
 #1
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [19]:
torch.manual_seed(123)
batch = torch.tensor([[6109,  3626,  6100,   345],      #1
                [6109,  1110,  6622,   257]])

model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.2102, -0.3511, -0.0213,  ..., -0.8458, -0.4478, -0.4313],
         [-0.0335, -0.8052, -0.1072,  ...,  0.0316, -0.2101, -0.6717],
         [ 0.4705,  0.3447, -0.0110,  ...,  0.1924,  0.7872,  0.2079],
         [-0.0227, -0.0384,  0.8313,  ...,  0.4261,  0.8232,  0.4191]],

        [[-0.2678, -0.3101,  0.6914,  ..., -0.2861, -0.7700, -0.5555],
         [-1.0854, -0.9912, -0.2878,  ..., -0.7238,  0.0845,  0.0212],
         [ 0.6291, -0.3692,  0.0452,  ..., -0.3714, -0.0321,  0.2745],
         [-0.1782, -0.6241, -0.5461,  ...,  0.8827,  0.4548,  0.3689]]],
       grad_fn=<UnsafeViewBackward0>)


# Training a model
## Loss calculation

In [20]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]
targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107, 588, 11311]])  #  " really like chocolate"]
with torch.no_grad():     #1
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1)     #2
print(probas.shape)
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:\n", token_ids)

text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

torch.Size([2, 3, 50257])
Token IDs:
 tensor([[[47050],
         [43457],
         [20421]],

        [[23736],
         [36574],
         [16258]]])
Text 1: tensor([1.5719e-05, 1.4242e-05, 1.1784e-05])
Text 2: tensor([3.7351e-06, 1.1969e-05, 2.8608e-05])


### Picking the probability of the correct next token

`probas` has shape `[batch, seq_len, vocab_size]`, here `[2, 3, 50257]`.

```python
probas[0, [0, 1, 2], [3626, 6100, 345]]
```

When you index with **two lists of the same length**, PyTorch **pairs them element by element**. It does not take every combination:

| Position | Correct next token | What it picks |
|---|---|---|
| 0 ("every") | 3626 ("effort") | `probas[0, 0, 3626]` |
| 1 ("effort") | 6100 ("moves") | `probas[0, 1, 6100]` |
| 2 ("moves") | 345 ("you") | `probas[0, 2, 345]` |

The result is **3 numbers**: the probability the model gave to the *correct* next token at each position. It is the same as:

```python
torch.tensor([probas[0, 0, 3626],
              probas[0, 1, 6100],
              probas[0, 2, 345]])
```

**Why this matters:** training tries to push these probabilities toward 1. The loss is the negative mean log of them, which is the cross-entropy loss:

```python
loss = -torch.log(target_probas).mean()
```

In [23]:
print('logits', logits.shape)
logits_flat = logits.flatten(0, 1)
targets_flat= targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

logits torch.Size([2, 3, 50257])
Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


In [24]:
torch.nn.functional.cross_entropy??

Signature:
torch.nn.functional.cross_entropy(
    input: torch.Tensor,
    target: torch.Tensor,
    weight: torch.Tensor | None = None,
    size_average: bool | None = None,
    ignore_index: int = -100,
    reduce: bool | None = None,
    reduction: str = 'mean',
    label_smoothing: float = 0.0,
) -> torch.Tensor
Source:   
def cross_entropy(
    input: Tensor,
    target: Tensor,
    weight: Tensor | None = None,
    size_average: bool | None = None,
    ignore_index: int = -100,
    reduce: bool | None = None,
    reduction: str = "mean",
    label_smoothing: float = 0.0,
) -> Tensor:
    r"""Compute the cross entropy loss between input logits and target.

    See :class:`~torch.nn.CrossEntropyLoss` for details.

    Args:
        input (Tensor) : Predicted unnormalized logits;
            see Shape section below for supported shapes.
        target (Tensor) : Ground truth class indices or class probabilities;
            see Shape section below for supported shapes.
        weigh